# BERT Toxic Comment Classifier - Production-Grade Pipeline

**Pipeline summary:**

| Area | Detail |
|---|---|
| **Architecture** | `AutoModelForSequenceClassification` (DeBERTa-v3-base) with multi-label head |
| **Focal Loss** | FocalLoss for imbalanced labels (used alone — NOT combined with `pos_weight`, to avoid double-compensating for imbalance) |
| **Text Cleaning** | `clean_text()` strips URLs/HTML/whitespace before tokenisation |
| **Mixed Precision** | `torch.cuda.amp` autocast + `GradScaler`, enabled automatically on CUDA |
| **Gradient Accumulation** | Simulates large batches without extra GPU memory |
| **Differential LR** | Lower LR for the backbone, higher LR for the classification head |
| **Cosine LR Schedule** | Cosine decay with warmup |
| **Checkpoint Saving** | Best model auto-saved mid-training (single canonical save dir) |
| **ROC Curves / AP** | Per-label ROC curves with AUC and average precision (real numbers only — no simulated baselines) |
| **Batch Inference** | `predict_batch()` returns a tidy DataFrame |
| **Streamlit App** | Auto-generated `app.py` that actually loads the model and runs inference |
| **Config Dataclass** | All hyperparameters in one typed `Config` |
| **Reproducibility** | Seeding covers PyTorch, NumPy, Python, and CUDA |


In [ ]:
# -- 0. Optional installs (run once) ------------------------------------------
# !pip install -q transformers datasets accelerate xgboost streamlit


In [ ]:
!nvidia-smi

In [ ]:
# -- 1. Imports ----------------------------------------------------------------
import re, os, csv, json, random, warnings, logging
from dataclasses import dataclass, field
from typing import Dict, List, Optional

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoConfig,
    get_cosine_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    average_precision_score, roc_curve,
)

plt.style.use('ggplot')
print('Imports complete.')


In [ ]:
# -- 2. Unified Config ---------------------------------------------------------
@dataclass
class Config:
    # Model
    model_name:         str   = 'microsoft/deberta-v3-base'
    max_len:            int   = 128
    hidden_dropout:     float = 0.3
    num_labels:         int   = 6

    # Training
    batch_size:         int   = 16
    grad_accum_steps:   int   = 2       # effective batch = batch_size * grad_accum_steps
    epochs:             int   = 4
    backbone_lr:        float = 2e-5    # differential LR: backbone gets the smaller rate
    head_lr:             float = 1e-4    # classification head gets a larger rate
    weight_decay:       float = 0.01
    warmup_frac:        float = 0.06
    max_grad_norm:      float = 1.0
    patience:           int   = 2

    # Loss
    use_focal_loss:     bool  = True
    focal_gamma:        float = 2.0

    # Threshold tuning
    # The search floor prevents the optimiser finding absurdly low
    # thresholds on skewed training data (e.g. Davidson fallback),
    # which would flag neutral text like 'good person' as toxic.
    min_threshold:      float = 0.35
    threshold_floor:    float = 0.35   # same value, used in search range
    # NOTE: we deliberately do NOT also apply pos_weight on top of focal loss -
    # focal loss already reweights toward minority/hard examples, and stacking
    # pos_weight on top of it was the root cause of the earlier NaN/instability
    # issues that forced FP32 + LR=1e-6 + grad-clip=0.5 band-aids.

    # Data
    nrows:              Optional[int] = 5000
    val_size:           float = 0.20
    seed:               int   = 42

    # Paths
    data_path:    str = 'train (1).csv'
    fallback_url: str = (
        'https://github.com/t-davidson/hate-speech-and-offensive-language'
        '/raw/master/data/labeled_data.csv'
    )
    save_dir:     str = './toxic_bert_model'

    # Labels
    label_cols: List[str] = field(default_factory=lambda: [
        'toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'
    ])

CFG = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = (DEVICE.type == 'cuda')
print(f'Device : {DEVICE}')
print(f'Mixed precision (AMP) : {USE_AMP}')
print(f'Effective batch size : {CFG.batch_size * CFG.grad_accum_steps}')


In [ ]:
# -- 3. Reproducibility --------------------------------------------------------
def set_seed(seed: int = CFG.seed) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed()
print(f'Seed set to {CFG.seed}')


In [ ]:
# -- 4. Text Cleaning ----------------------------------------------------------
_URL_RE     = re.compile(r'https?://\S+|www\.\S+')
_HTML_RE    = re.compile(r'<[^>]+' + '>')
_NEWLINE_RE = re.compile(r'[\r\n\t]+')
_MULTI_SP   = re.compile(r' {2,}')

def clean_text(text: str) -> str:
    """
    Light cleaning that keeps punctuation (important toxicity signal for BERT):
    1. Lower-case
    2. Strip URLs (noise in wiki-comments)
    3. Strip HTML tags
    4. Collapse newlines/tabs to a single space
    5. Collapse multiple spaces
    """
    text = text.lower()
    text = _URL_RE.sub(' ', text)
    text = _HTML_RE.sub(' ', text)
    text = _NEWLINE_RE.sub(' ', text)
    text = _MULTI_SP.sub(' ', text)
    return text.strip()

# Smoke test
_s = 'Check <b>this</b>: https://example.com\nYou are an idiot!!!'
print('Before:', _s)
print('After :', clean_text(_s))


In [ ]:
# -- 5. Data Loading & Preprocessing ------------------------------------------
def load_data(cfg: Config = CFG) -> pd.DataFrame:
    try:
        if os.path.exists(cfg.data_path):
            df = pd.read_csv(
                cfg.data_path, nrows=cfg.nrows, on_bad_lines='skip',
                engine='python', quoting=csv.QUOTE_NONE,
                escapechar='\\', encoding='latin-1',
            )
            print(f'Loaded local file: {cfg.data_path}')
        else:
            df = pd.read_csv(cfg.fallback_url, nrows=cfg.nrows, encoding='latin-1')
            print('Loaded fallback URL dataset.')
    except Exception as exc:
        raise RuntimeError(f'Data loading failed: {exc}') from exc

    # Normalise column names
    df.columns = [c.strip().replace('"', '') for c in df.columns]

    # Ensure comment_text column
    if 'comment_text' not in df.columns:
        for alias in ('tweet', 'text', 'comment'):
            if alias in df.columns:
                df = df.rename(columns={alias: 'comment_text'})
                break
        else:
            df = df.rename(columns={df.columns[1]: 'comment_text'})

    # Clean text
    df['comment_text'] = df['comment_text'].fillna('').astype(str).apply(clean_text)

    # Remap Davidson hate-speech fallback format
    # (class: 0=hate_speech, 1=offensive, 2=neither)
    if 'class' in df.columns and 'toxic' not in df.columns:
        df['toxic']         = (df['class'].isin([0, 1])).astype(int)
        df['severe_toxic']  = (df['class'] == 0).astype(int)
        df['obscene']       = (df['class'] == 1).astype(int)
        df['threat']        = 0
        df['insult']        = (df['class'] == 1).astype(int)
        df['identity_hate'] = (df['class'] == 0).astype(int)
        print('Fallback dataset: remapped class → 6-label schema.')
        # Balance fallback: downsample majority class to ~50% toxic
        # so the model doesn't learn a strong prior toward toxicity.
        _pos = df[df['toxic'] == 1]
        _neg = df[df['toxic'] == 0]
        if len(_pos) > len(_neg):
            _pos = _pos.sample(len(_neg), random_state=cfg.seed)
        elif len(_neg) > 2 * len(_pos):
            _neg = _neg.sample(2 * len(_pos), random_state=cfg.seed)
        df = pd.concat([_pos, _neg]).sample(frac=1, random_state=cfg.seed).reset_index(drop=True)
        print(f'After balancing — toxic: {df["toxic"].sum()}  non-toxic: {(df["toxic"]==0).sum()}')

    # Ensure label columns exist
    for col in cfg.label_cols:
        if col not in df.columns:
            df[col] = 0
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

    # Derived columns
    df['any_toxic']   = (df[cfg.label_cols].max(axis=1) > 0).astype(int)
    df['text_length'] = df['comment_text'].str.len()
    df['word_count']  = df['comment_text'].str.split().str.len()

    # Drop empty texts
    df = df[df['comment_text'].str.len() > 0].reset_index(drop=True)

    print(f'Shape    : {df.shape}')
    print(f'Toxic    : {df["any_toxic"].sum()}  |  Non-toxic : {(df["any_toxic"]==0).sum()}')
    print(f'Ratio    : {(df["any_toxic"]==0).sum() / max(df["any_toxic"].sum(),1):.1f}:1')
    return df

df = load_data()
df[['comment_text', 'any_toxic'] + CFG.label_cols].head(3)


In [ ]:
# -- 5b. Train / Val Split (shared by ALL models) ------------------------------
# Defined here — before tokenisation — so BiLSTM and classical-ML cells
# can reference the same indices even if the transformer tokenizer is skipped.
from sklearn.model_selection import train_test_split as _tts
import numpy as _np

label_data    = df[CFG.label_cols].values.astype(_np.float32)
binary_labels = df['any_toxic'].values.astype(_np.float32)

_indices = _np.arange(len(df))
train_idx, val_idx = _tts(
    _indices,
    test_size=CFG.val_size,
    random_state=CFG.seed,
    stratify=df['any_toxic'].values,
)

print(f'Train : {len(train_idx)}  |  Val : {len(val_idx)}')
print(f'Train toxic % : {label_data[train_idx, 0].mean()*100:.1f}%   '
      f'Val toxic % : {label_data[val_idx, 0].mean()*100:.1f}%')


In [ ]:
# -- 6. Exploratory Data Analysis (EDA) ---------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 6a Class balance
counts = df['any_toxic'].value_counts()
bars = axes[0,0].bar(['Non-toxic','Toxic'], counts.values,
                      color=['#4C9BE8','#E84C4C'], edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, counts.values):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, v+15,
                   f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9)
axes[0,0].set_title('Binary Class Balance', fontweight='bold')
axes[0,0].set_ylabel('Count')

# 6b Per-label counts
label_counts = df[CFG.label_cols].sum().sort_values(ascending=False)
sns.barplot(x=label_counts.index, y=label_counts.values, ax=axes[0,1], palette='viridis')
axes[0,1].set_title('Per-label Positive Counts', fontweight='bold')
axes[0,1].tick_params(axis='x', rotation=30)
for i, v in enumerate(label_counts.values):
    axes[0,1].text(i, v+3, str(int(v)), ha='center', fontsize=8)

# 6c Text length distribution by class
for label, grp in df.groupby('any_toxic'):
    axes[0,2].hist(grp['text_length'].clip(upper=500), bins=40, alpha=0.6,
                   label=['Non-toxic','Toxic'][label])
axes[0,2].set_title('Text Length by Class', fontweight='bold')
axes[0,2].set_xlabel('Characters (clipped at 500)')
axes[0,2].legend()

# 6d Word count boxplot
sns.boxplot(x='any_toxic', y='word_count', data=df[df['word_count']<200],
            ax=axes[1,0], palette=['#4C9BE8','#E84C4C'])
axes[1,0].set_title('Word Count Distribution', fontweight='bold')
axes[1,0].set_xticklabels(['Non-toxic','Toxic'])

# 6e Label correlation
sns.heatmap(df[CFG.label_cols].corr(), annot=True, cmap='coolwarm',
            fmt='.2f', vmin=-1, vmax=1, ax=axes[1,1], linewidths=0.5)
axes[1,1].set_title('Label Correlation Matrix', fontweight='bold')

# 6f Label co-occurrence
co = df[CFG.label_cols].T.dot(df[CFG.label_cols])
_co_arr = co.to_numpy().copy()
np.fill_diagonal(_co_arr, 0)
co = pd.DataFrame(_co_arr, index=co.index, columns=co.columns)
sns.heatmap(co, annot=True, fmt='d', cmap='YlOrRd', ax=axes[1,2], linewidths=0.5)
axes[1,2].set_title('Label Co-occurrence', fontweight='bold')

plt.suptitle('Exploratory Data Analysis', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('\nLabel prevalence (%):')
print((df[CFG.label_cols].mean() * 100).round(2).to_string())


In [ ]:
# -- 7. Focal Loss ---------------------------------------------------------
class FocalLoss(nn.Module):
    """
    Standard multi-label focal loss. Use this OR a pos_weight-based
    BCEWithLogitsLoss to handle imbalance - not both at once. Stacking both
    over-corrects: well-classified rare positives get an outsized gradient and
    training becomes unstable (this was the cause of the earlier NaN issues).
    """
    def __init__(self, gamma: float = 2.0, reduction: str = 'mean', eps: float = 1e-7):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.eps = eps

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        p_t = torch.where(targets == 1, probs, 1 - probs)
        focal = (1 - p_t + self.eps) ** self.gamma * bce

        if self.reduction == 'mean':
            return focal.mean()
        return focal.sum()

print('FocalLoss defined.')


In [ ]:
# -- 8. Tokenise + Stratified Split -------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

print('Tokenising (may take a moment)...')
encodings = tokenizer(
    df['comment_text'].tolist(),
    truncation=True,
    padding=True,
    max_length=CFG.max_len,
    return_tensors='pt',
)
print(f'Encodings shape : {encodings["input_ids"].shape}')


def make_encodings(idx):
    return {k: v[idx] for k, v in encodings.items()}

train_enc, val_enc = make_encodings(train_idx), make_encodings(val_idx)

print(f'Train : {len(train_idx)}  |  Val : {len(val_idx)}')


In [ ]:
# -- 9. Dataset & DataLoaders -------------------------------------------------
class ToxicDataset(Dataset):
    def __init__(self, encodings: Dict[str, torch.Tensor],
                 multi_labels: np.ndarray, binary_labels: np.ndarray):
        self.encodings     = encodings
        self.multi_labels  = multi_labels
        self.binary_labels = binary_labels

    def __len__(self) -> int:
        return len(self.binary_labels)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        item = {k: v[idx].clone().detach() for k, v in self.encodings.items()}
        item['multi_labels']  = torch.tensor(self.multi_labels[idx],  dtype=torch.float)
        item['binary_labels'] = torch.tensor(self.binary_labels[idx], dtype=torch.float)
        return item

train_ds = ToxicDataset(train_enc, label_data[train_idx], binary_labels[train_idx])
val_ds   = ToxicDataset(val_enc,   label_data[val_idx],   binary_labels[val_idx])

USE_PIN = (DEVICE.type == 'cuda')
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                          num_workers=0, pin_memory=USE_PIN)
val_loader   = DataLoader(val_ds, batch_size=CFG.batch_size * 2, shuffle=False,
                          num_workers=0, pin_memory=USE_PIN)

print(f'Batches - train: {len(train_loader)}  |  val: {len(val_loader)}')


In [ ]:
# -- 10. Model ------------------------------------------------------------------
# Swap CFG.model_name to 'distilbert-base-uncased' or 'roberta-base' to try other backbones.

def get_transformer_model(cfg: Config = CFG) -> nn.Module:
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=cfg.num_labels,
        problem_type='multi_label_classification',
    )
    return model.to(DEVICE)

model = get_transformer_model()
print(f'Loaded {CFG.model_name} with {model.config.num_labels} labels.')


In [ ]:
# -- 11. Loss, Optimiser & Scheduler -------------------------------------------
# Differential learning rates: small LR for the pretrained backbone, larger LR
# for the freshly-initialised classification head.
classifier_keywords = ['classifier', 'pooler']
head_params     = [p for n, p in model.named_parameters() if any(k in n for k in classifier_keywords)]
backbone_params = [p for n, p in model.named_parameters() if not any(k in n for k in classifier_keywords)]

optimizer = AdamW([
    {'params': backbone_params, 'lr': CFG.backbone_lr},
    {'params': head_params,     'lr': CFG.head_lr},
], weight_decay=CFG.weight_decay, eps=1e-8)

multi_loss_fn = FocalLoss(gamma=CFG.focal_gamma) if CFG.use_focal_loss else nn.BCEWithLogitsLoss()

total_steps  = (len(train_loader) // CFG.grad_accum_steps) * CFG.epochs
warmup_steps = max(1, int(total_steps * CFG.warmup_frac))
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
scaler = GradScaler(enabled=USE_AMP)

print(f'Backbone LR: {CFG.backbone_lr}  |  Head LR: {CFG.head_lr}')
print(f'Loss: {type(multi_loss_fn).__name__}  |  AMP: {USE_AMP}  |  Total steps: {total_steps}')


In [ ]:
# -- 12. Evaluation Helper -----------------------------------------------------
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate(loader: DataLoader, threshold: float = 0.5):
    model.eval()
    m_preds, m_labels, b_labels = [], [], []

    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)

        with autocast(enabled=USE_AMP):
            outputs = model(ids, attention_mask=mask)
            logits = outputs.logits

        m_preds.append(torch.sigmoid(logits.float()).cpu().numpy())
        m_labels.append(batch['multi_labels'].numpy())
        b_labels.append(batch['binary_labels'].numpy())

    mp = np.vstack(m_preds)
    ml = np.vstack(m_labels)
    bl = np.concatenate(b_labels)

    # For binary metrics, we consider a sample toxic if ANY label is flagged
    bp = mp.max(axis=1)

    m_true = (ml > 0.5).astype(int)
    m_pred = (mp > threshold).astype(int)
    b_true = (bl > 0.5).astype(int)
    b_pred = (bp > threshold).astype(int)

    macro_f1 = f1_score(m_true, m_pred, average='macro',  zero_division=0)
    bin_f1   = f1_score(b_true, b_pred, average='binary', zero_division=0)
    return macro_f1, bin_f1, mp, bp, ml, bl

print('evaluate() ready.')


In [ ]:
# -- 13. Training Loop ----------------------------------------------------------
os.makedirs(CFG.save_dir, exist_ok=True)
CKPT_PATH = os.path.join(CFG.save_dir, 'best_checkpoint.pt')

history = {'train_loss': [], 'val_macro_f1': [], 'val_bin_f1': []}
best_f1 = -1.0
no_improve = 0

for epoch in range(CFG.epochs):
    model.train()
    epoch_loss = 0.0
    n_steps = 0
    optimizer.zero_grad()
    loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{CFG.epochs}', leave=False)

    for step, batch in enumerate(loop):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        m_lbl = batch['multi_labels'].to(DEVICE)

        with autocast(enabled=USE_AMP):
            logits = model(ids, attention_mask=mask).logits
            loss = multi_loss_fn(logits, m_lbl) / CFG.grad_accum_steps

        if torch.isnan(loss):
            print(f'\n[Warning] NaN loss at epoch {epoch+1} step {step}, skipping batch.')
            optimizer.zero_grad()
            continue

        scaler.scale(loss).backward()

        is_last_batch = (step + 1) == len(train_loader)
        if (step + 1) % CFG.grad_accum_steps == 0 or is_last_batch:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        epoch_loss += loss.item() * CFG.grad_accum_steps
        n_steps += 1
        loop.set_postfix(loss=f'{loss.item()*CFG.grad_accum_steps:.4f}')

    avg_loss = epoch_loss / max(1, n_steps)
    macro_f1, bin_f1, *_ = evaluate(val_loader)

    history['train_loss'].append(avg_loss)
    history['val_macro_f1'].append(macro_f1)
    history['val_bin_f1'].append(bin_f1)

    print(f'Epoch {epoch+1} | Loss: {avg_loss:.4f} | Val Macro-F1: {macro_f1:.4f} | Val Bin-F1: {bin_f1:.4f}')

    if macro_f1 > best_f1:
        best_f1 = macro_f1
        no_improve = 0
        torch.save(model.state_dict(), CKPT_PATH)
        print('  --> Saved new best model')
    else:
        no_improve += 1
        if no_improve >= CFG.patience:
            print('Early stopping triggered.')
            break

print(f'\nTraining finished. Best Val Macro-F1: {best_f1:.4f}')


In [ ]:
# -- 14. Training Curves -------------------------------------------------------
epochs_ran = list(range(1, len(history['train_loss']) + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(epochs_ran, history['train_loss'], marker='o', color='#E84C4C', label='Train Loss')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(epochs_ran, history['val_macro_f1'], marker='o', color='#4C9BE8', label='Macro-F1')
axes[1].plot(epochs_ran, history['val_bin_f1'], marker='s', color='#E8A54C', label='Binary F1')
axes[1].set_title('Validation F1 Metrics', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# -- 15. Per-label Threshold Tuning & Full Evaluation -------------------------
_, _, all_preds, bin_preds, all_labels, bin_labels = evaluate(val_loader)

print('\n=== Per-label Threshold Optimisation ===\n')
best_thresholds  = []
threshold_search = np.linspace(CFG.threshold_floor, 0.90, 100)
rows = []

for i, name in enumerate(CFG.label_cols):
    y_true = (all_labels[:, i] > 0.5).astype(int)
    y_prob = all_preds[:, i]

    if len(np.unique(y_true)) > 1:
        f1s = [f1_score(y_true, (y_prob > t).astype(int), zero_division=0) for t in threshold_search]
        best_t = max(threshold_search[np.argmax(f1s)], CFG.min_threshold)
        auc = roc_auc_score(y_true, y_prob)
        ap  = average_precision_score(y_true, y_prob)
    else:
        best_t = max(0.5, CFG.min_threshold)
        auc = 0.0
        ap = 0.0

    best_thresholds.append(float(best_t))
    rows.append({'label': name, 'threshold': round(best_t, 3), 'ROC-AUC': round(auc, 4), 'Avg-Precision': round(ap, 4)})

print(pd.DataFrame(rows).to_string(index=False))

# Binary threshold for 'any_toxic'
bin_true = (bin_labels > 0.5).astype(int)
bin_f1s = [f1_score(bin_true, (bin_preds > t).astype(int), zero_division=0) for t in threshold_search]
best_bin_t = max(threshold_search[np.argmax(bin_f1s)], CFG.min_threshold)

print(f'\nBest Binary Threshold: {best_bin_t:.3f}')


In [ ]:
# -- 16. Confusion Matrices ---------------------------------------------------
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

y_pred_bin = (bin_preds > best_bin_t).astype(int)

for i, name in enumerate(CFG.label_cols):
    y_true_i = (all_labels[:, i] > 0.5).astype(int)
    y_pred_i = (all_preds[:, i] > best_thresholds[i]).astype(int)
    cm = confusion_matrix(y_true_i, y_pred_i, labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=[f'Non-{name}', name]).plot(
        ax=axes[i], colorbar=False, cmap='Blues'
    )
    axes[i].set_title(name, fontweight='bold')

cm_bin = confusion_matrix(bin_true, y_pred_bin, labels=[0, 1])
ConfusionMatrixDisplay(cm_bin, display_labels=['Non-toxic', 'Toxic']).plot(
    ax=axes[6], colorbar=False, cmap='Oranges'
)
axes[6].set_title('Binary: Toxic vs Non-toxic', fontweight='bold')
axes[7].axis('off')

plt.suptitle('Confusion Matrices (Tuned Thresholds)', fontsize=13,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# -- 17. ROC Curves -------------------------------------------------------------
# NOTE: only real, measured curves are plotted here. The previous version
# overlaid fabricated "benchmark" ROC curves for DistilBERT/BiLSTM/TextCNN that
# were synthetically generated to hit a target AUC rather than measured from
# real models - that's misleading and has been removed. If you want a genuine
# comparison, train those baselines (see the classical-ML and BiLSTM cells
# below) and plot their real roc_curve() output instead.
fig, ax = plt.subplots(figsize=(9, 7))
palette = plt.cm.get_cmap('tab10').colors

for i, name in enumerate(CFG.label_cols):
    y_true_i = (all_labels[:, i] > 0.5).astype(int)
    if len(np.unique(y_true_i)) < 2:
        continue
    fpr, tpr, _ = roc_curve(y_true_i, all_preds[:, i])
    auc_val = roc_auc_score(y_true_i, all_preds[:, i])
    ax.plot(fpr, tpr, color=palette[i % 10], alpha=0.6, lw=1.5,
            label=f'{name} (AUC={auc_val:.2f})')

fpr_b, tpr_b, _ = roc_curve(bin_true, bin_preds)
bin_auc_val = roc_auc_score(bin_true, bin_preds)
ax.plot(fpr_b, tpr_b, 'black', lw=3, label=f'Binary toxicity (AUC={bin_auc_val:.3f})')

ax.plot([0, 1], [0, 1], 'grey', lw=1, linestyle=':')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves - {CFG.model_name}', fontweight='bold', fontsize=14)
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# -- 18. Save Model, Tokeniser & Thresholds ------------------------------------
os.makedirs(CFG.save_dir, exist_ok=True)

# Save the full HuggingFace model (backbone + classifier head) and tokenizer
model.save_pretrained(CFG.save_dir)
tokenizer.save_pretrained(CFG.save_dir)

# Save the calibrated thresholds for later use
# Enforce minimum floor before saving so the app never loads
# dangerously low thresholds from a skewed-data training run.
thresholds_dict = {
    name: float(max(t, CFG.min_threshold))
    for name, t in zip(CFG.label_cols, best_thresholds)
}
thresholds_dict['binary'] = float(max(best_bin_t, CFG.min_threshold))
print('Saved thresholds:', {k: round(v,3) for k,v in thresholds_dict.items()})

with open(os.path.join(CFG.save_dir, 'thresholds.json'), 'w') as fh:
    json.dump(thresholds_dict, fh, indent=2)

# Save the training configuration for reproducibility
config_dict = {
    'model_name': CFG.model_name,
    'max_len': CFG.max_len,
    'label_cols': CFG.label_cols,
    'backbone_lr': CFG.backbone_lr,
    'head_lr': CFG.head_lr,
    'use_focal_loss': CFG.use_focal_loss,
    'amp': USE_AMP,
}
with open(os.path.join(CFG.save_dir, 'pipeline_config.json'), 'w') as fh:
    json.dump(config_dict, fh, indent=2)

print(f'Pipeline artifacts saved to "{CFG.save_dir}"')
print('Contents:', os.listdir(CFG.save_dir))


In [ ]:
# -- 19. Reload from Checkpoint -------------------------------------------------
def load_model_from_checkpoint(save_dir: str = CFG.save_dir):
    """Reconstruct the model from saved artifacts."""
    m = AutoModelForSequenceClassification.from_pretrained(save_dir)
    return m.to(DEVICE).eval()

print('load_model_from_checkpoint() ready.')


In [ ]:
# -- 20. Single-text Inference ---------------------------------------------------
@torch.no_grad()
def predict_toxicity(
    text: str,
    multi_thresholds: Optional[List[float]] = None,
    bin_threshold: Optional[float] = None,
) -> Dict:
    if multi_thresholds is None: multi_thresholds = best_thresholds
    if bin_threshold is None: bin_threshold = best_bin_t

    model.eval()
    cleaned = clean_text(text)
    enc = tokenizer(cleaned, return_tensors='pt', truncation=True, padding=True, max_length=CFG.max_len)
    inputs = {k: v.to(DEVICE) for k, v in enc.items()}

    outputs = model(**inputs)
    logits = outputs.logits[0]
    m_probs = torch.sigmoid(logits).cpu().numpy()
    b_prob = float(np.max(m_probs))

    return {
        'toxic' : bool(b_prob > bin_threshold),
        'score' : round(b_prob, 4),
        'labels': {
            name: {'prob': round(float(p), 4), 'flagged': bool(p > t)}
            for name, p, t in zip(CFG.label_cols, m_probs, multi_thresholds)
        },
    }

def print_prediction(text: str) -> None:
    r = predict_toxicity(text)
    verdict = 'TOXIC' if r['toxic'] else 'NON-TOXIC'
    bar = '#' * int(r['score'] * 20) + '.' * (20 - int(r['score'] * 20))
    print(f'\nText    : {text[:90]}')
    print(f'Verdict : {verdict}  score={r["score"]:.4f}  [{bar}]')
    for name, info in r['labels'].items():
        flag = '^' if info['flagged'] else ' '
        print(f'  {flag} {name:15} {info["prob"]:.4f}')

demo_texts = [
    "I really appreciate your help with this project, thank you!",
    "You are so stupid and I hate everything about you.",
    "This is a lovely sunny day.",
    "Go kill yourself, nobody likes you.",
    "The president made an announcement about the budget.",
    "You filthy idiot, I hope you suffer."
]

for t in demo_texts:
    print_prediction(t)


In [ ]:
# -- 21. Batch Inference ---------------------------------------------------------
@torch.no_grad()
def predict_batch(texts: List[str], batch_size: int = 32) -> pd.DataFrame:
    """
    Run inference on a list of texts.
    Returns a DataFrame with columns: text, toxic, score, + one prob per label.
    """
    model.eval()
    records = []

    for start in range(0, len(texts), batch_size):
        chunk  = [clean_text(t) for t in texts[start:start + batch_size]]
        enc    = tokenizer(chunk, truncation=True, padding=True,
                            max_length=CFG.max_len, return_tensors='pt')
        inputs = {k: v.to(DEVICE) for k, v in enc.items()}

        outputs = model(**inputs)
        logits = outputs.logits

        m_probs = torch.sigmoid(logits).cpu().numpy()
        b_probs = np.max(m_probs, axis=1)

        for i, text in enumerate(texts[start:start + batch_size]):
            row = {'text' : text[:120],
                   'toxic': bool(b_probs[i] > best_bin_t),
                   'score': round(float(b_probs[i]), 4)}
            for j, name in enumerate(CFG.label_cols):
                row[name] = round(float(m_probs[i, j]), 4)
            records.append(row)

    return pd.DataFrame(records)

# Demo
batch_df = predict_batch(demo_texts)
print(batch_df.to_string(index=False))


In [ ]:
# -- 22. Write Streamlit Demo App (fixed threshold floor + calibration notice) --
APP_CODE = '''
"""
Toxic Comment Classifier — Streamlit App (Fixed)

Quick-fix applied on top of the existing saved model:
  1. Minimum threshold floor of 0.55 applied to all labels so that
     neutral text like "good person" is never flagged.
  2. Thresholds are shown next to every score so the user can see
     exactly why something is or isn't flagged.
  3. A recalibration note warns that scores near 0.5 indicate a
     miscalibrated model (trained on skewed data) and that retraining
     on the Jigsaw dataset with the fixed pipeline will resolve this.

Permanent fix: retrain using bert_toxic_v5_clean.ipynb with the Jigsaw
train.csv dataset (NOT the Davidson fallback). The notebook now includes
balanced data loading, min_threshold=0.35 in Config, and threshold search
starting at 0.35, which together prevent the model learning a toxic-bias prior.
"""

import streamlit as st
import torch, json, re, os
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ── Constants ──────────────────────────────────────────────────────────────────
LABEL_COLS   = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
SAVE_DIR     = "./toxic_bert_model"
MAX_LEN      = 128

# Minimum threshold floor applied regardless of what thresholds.json says.
# This is a post-hoc correction for models trained on skewed data.
# When you retrain on balanced Jigsaw data the saved thresholds will be
# sensible (≥ 0.45) and this floor will have no effect.
MIN_THRESHOLD_FLOOR = 0.55

# ── Asset loading (cached) ─────────────────────────────────────────────────────
@st.cache_resource
def load_assets():
    tok   = AutoTokenizer.from_pretrained(SAVE_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
    model.eval()

    with open(os.path.join(SAVE_DIR, "thresholds.json")) as f:
        raw_thresh = json.load(f)

    # Apply floor: raise any threshold that's unreasonably low
    thresh = {
        k: max(float(v), MIN_THRESHOLD_FLOOR)
        for k, v in raw_thresh.items()
    }

    original_low = {k: v for k, v in raw_thresh.items()
                    if float(v) < MIN_THRESHOLD_FLOOR}
    return tok, model, thresh, original_low

tokenizer, model, thresholds, original_low = load_assets()

# ── Text cleaning (must match training pipeline) ───────────────────────────────
_URL_RE  = re.compile(r"https?://\S+|www\.\S+")
_HTML_RE = re.compile(r"<[^>]+>")

def clean(text: str) -> str:
    text = text.lower()
    text = _URL_RE.sub(" ", text)
    text = _HTML_RE.sub(" ", text)
    return re.sub(r" {2,}", " ", text).strip()

# ── Inference ──────────────────────────────────────────────────────────────────
@torch.no_grad()
def predict(text: str):
    cleaned = clean(text)
    enc     = tokenizer(cleaned, return_tensors="pt",
                        truncation=True, max_length=MAX_LEN)
    logits  = model(**enc).logits[0]
    probs   = torch.sigmoid(logits.float()).numpy()
    bin_score = float(np.max(probs))
    label_probs = {name: float(p) for name, p in zip(LABEL_COLS, probs)}
    return bin_score, label_probs

# ── Page layout ────────────────────────────────────────────────────────────────
st.set_page_config(page_title="Toxic Comment Classifier", page_icon="🛡️",
                   layout="centered")

st.title("🛡️ Toxic Comment Classifier")
st.write("Enter a comment below to evaluate multi-label classification predictions.")

# Calibration warning if thresholds were raised by the floor
if original_low:
    st.warning(
        f"⚠️ **Model calibration notice:** "
        f"The saved thresholds for "
        f"**{', '.join(original_low.keys())}** were unusually low "
        f"({', '.join(f'{v:.2f}' for v in original_low.values())}), "
        f"which caused neutral text to be flagged as toxic. "
        f"A minimum floor of **{MIN_THRESHOLD_FLOOR}** has been applied. "
        f"**Permanent fix:** retrain with the Jigsaw dataset using "
        f"`bert_toxic_v5_clean.ipynb` (fixed pipeline).",
        icon="⚠️",
    )

# ── Input ──────────────────────────────────────────────────────────────────────
st.markdown("**Comment Content Evaluation Window**")
user_input = st.text_area("", height=130, placeholder="Type a comment here…",
                           label_visibility="collapsed")

analyse = st.button("Analyse Text Content", type="primary")

if analyse and user_input.strip():
    bin_score, label_probs = predict(user_input)

    bin_thresh = thresholds.get("binary", MIN_THRESHOLD_FLOOR)
    verdict    = "TOXIC" if bin_score > bin_thresh else "NON-TOXIC"

    # Verdict banner
    color = "#5c0000" if verdict == "TOXIC" else "#003300"
    st.markdown(
        f'<div style="background:{color};padding:14px 18px;border-radius:8px;margin:12px 0;">'
        f'<span style="font-size:1.25rem;font-weight:700;">'
        f'Verdict: {verdict}</span>'
        f'<span style="float:right;opacity:.75;">'
        f'Score: {bin_score:.3f} / Threshold: {bin_thresh:.2f}</span>'
        f'</div>',
        unsafe_allow_html=True,
    )

    st.divider()
    st.subheader("Sub-Category Probabilities Breakdown")

    for name, prob in label_probs.items():
        thresh  = thresholds.get(name, MIN_THRESHOLD_FLOOR)
        flagged = prob > thresh

        col1, col2 = st.columns([1, 3])
        with col1:
            label_text = name.replace("_", " ").title()
            st.markdown(f"**{label_text}**")
            if flagged:
                st.caption("⚠️ Flagged")
            else:
                st.caption("✅ Clear")
        with col2:
            st.progress(min(prob, 1.0))
            st.caption(f"Score: {prob:.3f} / Threshold: {thresh:.2f}")

    # Show cleaned text used for inference
    with st.expander("🔍 Cleaned input sent to model"):
        st.code(clean(user_input))

elif analyse and not user_input.strip():
    st.info("Please enter some text before clicking Analyse.")

'''

with open('app.py', 'w') as f:
    f.write(APP_CODE)
print('app.py written. Run with: streamlit run app.py')


## Further Improvements

| Technique | How |
|---|---|
| **K-Fold CV** | Wrap training in 5-fold loop; ensemble soft predictions with `np.mean` |
| **Oversampling** | Use `imbalanced-learn` `RandomOverSampler` on minority labels before tokenisation |
| **Back-translation** | Translate toxic samples to another language and back to augment minority class |
| **Pseudo-labelling** | Add high-confidence model predictions on unlabelled data back into training |
| **Label smoothing** | Replace hard 0/1 targets with 0.1/0.9 to reduce overconfidence |
| **SHAP / Captum** | Token-level attributions via `transformers-interpret` or `captum` |
| **ONNX export** | `torch.onnx.export(model, ...)` for fast CPU-only deployment |
| **Quantisation** | `torch.quantization.quantize_dynamic` - 4x smaller, ~2x faster on CPU |
| **Confidence calibration** | Fit `sklearn.calibration.IsotonicRegression` on val probs post-training |


## Multi-Transformer Comparison

Run DeBERTa-v3-base, RoBERTa-base and DistilBERT through the same training
pipeline and collect their held-out metrics. Each model gets its own tokeniser,
is trained independently, and is scored on the same val split.

> **Runtime note:** Training three full transformers sequentially is time-intensive
> on Colab's free GPU tier. To speed things up you can reduce `CFG.epochs` or
> `CFG.nrows` before running this section, or comment out one of the model names.


In [ ]:
# -- Multi-transformer helpers --------------------------------------------------
import copy, gc

def tokenise_for_model(model_name: str, texts):
    """Return encodings using the tokeniser matching model_name."""
    tok = AutoTokenizer.from_pretrained(model_name)
    enc = tok(
        texts, truncation=True, padding=True,
        max_length=CFG.max_len, return_tensors='pt',
    )
    return tok, enc


def build_loaders_for_encodings(enc, label_data, binary_labels, train_idx, val_idx):
    """Build DataLoaders given fresh encodings for a specific model."""
    def _make(idx): return {k: v[idx] for k, v in enc.items()}

    tr_ds = ToxicDataset(_make(train_idx), label_data[train_idx], binary_labels[train_idx])
    va_ds = ToxicDataset(_make(val_idx),   label_data[val_idx],   binary_labels[val_idx])

    USE_PIN = (DEVICE.type == 'cuda')
    tr_l = DataLoader(tr_ds, batch_size=CFG.batch_size, shuffle=True,  num_workers=0, pin_memory=USE_PIN)
    va_l = DataLoader(va_ds, batch_size=CFG.batch_size*2, shuffle=False, num_workers=0, pin_memory=USE_PIN)
    return tr_l, va_l


def train_transformer(model_name: str, train_loader, val_loader,
                      label_data, binary_labels, val_idx,
                      n_epochs: int = CFG.epochs) -> dict:
    """
    Train a HuggingFace transformer for n_epochs and return a results dict.
    The model is discarded after scoring to free GPU memory.
    """
    print(f'\n{"="*60}')
    print(f'  Training: {model_name}')
    print(f'{"="*60}')

    m = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=CFG.num_labels,
        problem_type='multi_label_classification',
    ).to(DEVICE)

    # Differential LR
    ck = ['classifier', 'pooler']
    head_p     = [p for n, p in m.named_parameters() if any(k in n for k in ck)]
    backbone_p = [p for n, p in m.named_parameters() if not any(k in n for k in ck)]

    opt = AdamW([
        {'params': backbone_p, 'lr': CFG.backbone_lr},
        {'params': head_p,     'lr': CFG.head_lr},
    ], weight_decay=CFG.weight_decay, eps=1e-8)

    loss_fn = FocalLoss(gamma=CFG.focal_gamma) if CFG.use_focal_loss else nn.BCEWithLogitsLoss()
    total_steps  = (len(train_loader) // CFG.grad_accum_steps) * n_epochs
    warmup_steps = max(1, int(total_steps * CFG.warmup_frac))
    sched  = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)
    scaler = GradScaler(enabled=USE_AMP)

    best_macro_f1 = -1.0
    best_state    = None
    history = {'train_loss': [], 'val_macro_f1': [], 'val_bin_f1': []}

    for epoch in range(n_epochs):
        m.train()
        epoch_loss, n_steps = 0.0, 0
        opt.zero_grad()

        for step, batch in enumerate(tqdm(train_loader, desc=f'  Epoch {epoch+1}/{n_epochs}', leave=False)):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            mlbl = batch['multi_labels'].to(DEVICE)

            with autocast(enabled=USE_AMP):
                logits = m(ids, attention_mask=mask).logits
                loss   = loss_fn(logits, mlbl) / CFG.grad_accum_steps

            if torch.isnan(loss):
                opt.zero_grad()
                continue

            scaler.scale(loss).backward()
            is_last = (step + 1) == len(train_loader)
            if (step + 1) % CFG.grad_accum_steps == 0 or is_last:
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(m.parameters(), CFG.max_grad_norm)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad()

            epoch_loss += loss.item() * CFG.grad_accum_steps
            n_steps    += 1

        avg_loss = epoch_loss / max(1, n_steps)

        # Evaluate
        m.eval()
        m_preds, m_labels_list, b_labels_list = [], [], []
        with torch.no_grad():
            for batch in val_loader:
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                with autocast(enabled=USE_AMP):
                    logits = m(ids, attention_mask=mask).logits
                m_preds.append(torch.sigmoid(logits.float()).cpu().numpy())
                m_labels_list.append(batch['multi_labels'].numpy())
                b_labels_list.append(batch['binary_labels'].numpy())

        mp = np.vstack(m_preds)
        ml = np.vstack(m_labels_list)

        m_true   = (ml > 0.5).astype(int)
        m_pred   = (mp > 0.5).astype(int)
        b_true   = ((np.concatenate(b_labels_list)) > 0.5).astype(int)
        b_pred   = ((np.max(mp, axis=1)) > 0.5).astype(int)

        macro_f1 = f1_score(m_true, m_pred, average='macro',  zero_division=0)
        bin_f1   = f1_score(b_true, b_pred, average='binary', zero_division=0)

        history['train_loss'].append(avg_loss)
        history['val_macro_f1'].append(macro_f1)
        history['val_bin_f1'].append(bin_f1)
        print(f'  Epoch {epoch+1} | loss={avg_loss:.4f} | macro-F1={macro_f1:.4f} | bin-F1={bin_f1:.4f}')

        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            best_state    = copy.deepcopy(m.state_dict())

    # Restore best and do final eval
    m.load_state_dict(best_state)
    m.eval()
    final_preds = []
    with torch.no_grad():
        for batch in val_loader:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            with autocast(enabled=USE_AMP):
                logits = m(ids, attention_mask=mask).logits
            final_preds.append(torch.sigmoid(logits.float()).cpu().numpy())

    fp = np.vstack(final_preds)
    b_prob = np.max(fp, axis=1)

    ml_all = label_data[val_idx]
    bl_all = binary_labels[val_idx]
    m_true = (ml_all > 0.5).astype(int)
    b_true = (bl_all  > 0.5).astype(int)

    macro_f1_final = f1_score(m_true, (fp > 0.5).astype(int), average='macro',  zero_division=0)
    bin_f1_final   = f1_score(b_true, (b_prob > 0.5).astype(int), average='binary', zero_division=0)
    try:
        roc_auc = roc_auc_score(b_true, b_prob)
    except Exception:
        roc_auc = float('nan')

    # Free GPU memory
    del m, best_state, opt, sched, scaler
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    return {
        'model': model_name.split('/')[-1],
        'best_macro_f1': round(macro_f1_final, 4),
        'best_bin_f1':   round(bin_f1_final, 4),
        'roc_auc':       round(roc_auc, 4),
        'history':       history,
    }


print('Multi-transformer helpers ready.')


In [ ]:
# -- Run multi-transformer comparison ------------------------------------------
# Models to compare. Comment out any you want to skip.
TRANSFORMER_NAMES = [
    'microsoft/deberta-v3-base',
    'FacebookAI/roberta-base',
    'distilbert-base-uncased',
]

transformer_results = {}  # name → result dict

for model_name in TRANSFORMER_NAMES:
    # Each model needs its own tokeniser and encodings
    tok_i, enc_i = tokenise_for_model(model_name, df['comment_text'].tolist())
    tr_l_i, va_l_i = build_loaders_for_encodings(
        enc_i, label_data, binary_labels, train_idx, val_idx
    )

    result = train_transformer(
        model_name, tr_l_i, va_l_i,
        label_data, binary_labels, val_idx,
        n_epochs=CFG.epochs,
    )
    transformer_results[result['model']] = result

    # Free tokeniser encodings immediately
    del tok_i, enc_i, tr_l_i, va_l_i
    import gc; gc.collect()

print('\nAll transformer models trained.')


In [ ]:
# -- Plot learning curves for all transformers ----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
palette = ['#E84C4C', '#4C9BE8', '#4CE874']

for (name, res), color in zip(transformer_results.items(), palette):
    epochs_x = range(1, len(res['history']['train_loss']) + 1)
    axes[0].plot(epochs_x, res['history']['train_loss'],  marker='o', color=color, label=name)
    axes[1].plot(epochs_x, res['history']['val_macro_f1'], marker='s', color=color, label=name)

axes[0].set_title('Training Loss by Epoch', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(fontsize=8)
axes[1].set_title('Val Macro-F1 by Epoch',  fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


## BiLSTM Baseline

A BiLSTM trained with the same FocalLoss and val split as the transformers,
so the final comparison table is a genuine apples-to-apples benchmark.
Unlike the transformer blocks above this one builds its own character-free
word vocabulary from the training comments (no pre-trained embeddings needed
unless you want to plug in GloVe/FastText weights later).


In [ ]:
# -- BiLSTM: vocabulary from training text ----------------------------------------
from collections import Counter

def build_vocab(texts, max_vocab=30_000):
    counter = Counter()
    for t in texts:
        counter.update(t.split())
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in counter.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab

def encode_texts(texts, vocab, max_len=128):
    unk = vocab['<UNK>']
    ids = np.zeros((len(texts), max_len), dtype=np.int64)
    for i, text in enumerate(texts):
        tokens = text.split()[:max_len]
        for j, tok in enumerate(tokens):
            ids[i, j] = vocab.get(tok, unk)
    return ids

train_texts = df.iloc[train_idx]['comment_text'].tolist()
lstm_vocab  = build_vocab(train_texts)
print(f'Vocabulary size : {len(lstm_vocab):,}')

all_texts_encoded = encode_texts(df['comment_text'].tolist(), lstm_vocab)
print(f'Encoded shape   : {all_texts_encoded.shape}')


In [ ]:
# -- BiLSTM: Dataset & DataLoaders ----------------------------------------
class LSTMDataset(Dataset):
    def __init__(self, ids: np.ndarray, multi_labels: np.ndarray, binary_labels: np.ndarray):
        self.ids           = torch.from_numpy(ids)
        self.multi_labels  = torch.from_numpy(multi_labels).float()
        self.binary_labels = torch.from_numpy(binary_labels).float()

    def __len__(self): return len(self.binary_labels)

    def __getitem__(self, idx):
        return {
            'input_ids':     self.ids[idx],
            'multi_labels':  self.multi_labels[idx],
            'binary_labels': self.binary_labels[idx],
        }

USE_PIN = (DEVICE.type == 'cuda')

lstm_train_ds = LSTMDataset(all_texts_encoded[train_idx], label_data[train_idx], binary_labels[train_idx])
lstm_val_ds   = LSTMDataset(all_texts_encoded[val_idx],   label_data[val_idx],   binary_labels[val_idx])

lstm_train_loader = DataLoader(lstm_train_ds, batch_size=64, shuffle=True,  num_workers=0, pin_memory=USE_PIN)
lstm_val_loader   = DataLoader(lstm_val_ds,   batch_size=128, shuffle=False, num_workers=0, pin_memory=USE_PIN)

print(f'Train batches: {len(lstm_train_loader)}  |  Val batches: {len(lstm_val_loader)}')


In [ ]:
# -- BiLSTM: Model definition --------------------------------------------------
class BiLSTMToxicClassifier(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int = 100,
                 hidden_dim: int = 128, num_labels: int = 6):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            batch_first=True, bidirectional=True, num_layers=2,
            dropout=0.3,
        )
        self.dropout = nn.Dropout(0.3)
        self.fc      = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, input_ids):
        x = self.dropout(self.embedding(input_ids))   # (B, L, E)
        out, _ = self.lstm(x)                          # (B, L, 2H)
        pooled, _ = torch.max(out, dim=1)              # max-pool over time
        return self.fc(self.dropout(pooled))            # (B, num_labels)

lstm_model = BiLSTMToxicClassifier(
    vocab_size=len(lstm_vocab),
    embed_dim=100,
    hidden_dim=128,
    num_labels=CFG.num_labels,
).to(DEVICE)

total_params = sum(p.numel() for p in lstm_model.parameters())
print(f'BiLSTM parameters: {total_params:,}')


In [ ]:
# -- BiLSTM: Training loop ------------------------------------------------------
LSTM_EPOCHS = 5
lstm_loss_fn = FocalLoss(gamma=CFG.focal_gamma)
lstm_opt     = torch.optim.Adam(lstm_model.parameters(), lr=1e-3)

lstm_history = {'train_loss': [], 'val_macro_f1': [], 'val_bin_f1': []}
best_lstm_f1    = -1.0
best_lstm_state = None

for epoch in range(LSTM_EPOCHS):
    lstm_model.train()
    epoch_loss, n_steps = 0.0, 0

    for batch in tqdm(lstm_train_loader, desc=f'BiLSTM Epoch {epoch+1}/{LSTM_EPOCHS}', leave=False):
        ids  = batch['input_ids'].to(DEVICE)
        mlbl = batch['multi_labels'].to(DEVICE)

        logits = lstm_model(ids)
        loss   = lstm_loss_fn(logits, mlbl)

        lstm_opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_model.parameters(), 1.0)
        lstm_opt.step()

        epoch_loss += loss.item()
        n_steps    += 1

    # Validation
    lstm_model.eval()
    preds_v, labels_v = [], []
    with torch.no_grad():
        for batch in lstm_val_loader:
            logits = lstm_model(batch['input_ids'].to(DEVICE))
            preds_v.append(torch.sigmoid(logits).cpu().numpy())
            labels_v.append(batch['multi_labels'].numpy())

    mp = np.vstack(preds_v)
    ml = np.vstack(labels_v)
    bl = binary_labels[val_idx]

    m_true = (ml > 0.5).astype(int)
    m_pred = (mp > 0.5).astype(int)
    b_true = (bl > 0.5).astype(int)
    b_pred = (np.max(mp, axis=1) > 0.5).astype(int)

    macro_f1 = f1_score(m_true, m_pred, average='macro',  zero_division=0)
    bin_f1   = f1_score(b_true, b_pred, average='binary', zero_division=0)
    avg_loss = epoch_loss / max(1, n_steps)

    lstm_history['train_loss'].append(avg_loss)
    lstm_history['val_macro_f1'].append(macro_f1)
    lstm_history['val_bin_f1'].append(bin_f1)
    print(f'BiLSTM Epoch {epoch+1} | loss={avg_loss:.4f} | macro-F1={macro_f1:.4f} | bin-F1={bin_f1:.4f}')

    if macro_f1 > best_lstm_f1:
        best_lstm_f1    = macro_f1
        best_lstm_state = {k: v.clone() for k, v in lstm_model.state_dict().items()}

lstm_model.load_state_dict(best_lstm_state)
print(f'\nBiLSTM best val macro-F1: {best_lstm_f1:.4f}')


In [ ]:
# -- BiLSTM: Final score (for comparison table) --------------------------------
lstm_model.eval()
lstm_preds_all = []
with torch.no_grad():
    for batch in lstm_val_loader:
        logits = lstm_model(batch['input_ids'].to(DEVICE))
        lstm_preds_all.append(torch.sigmoid(logits).cpu().numpy())

lstm_mp = np.vstack(lstm_preds_all)
lstm_b_prob = np.max(lstm_mp, axis=1)
lstm_b_true = (binary_labels[val_idx] > 0.5).astype(int)

lstm_final_macro = f1_score((label_data[val_idx] > 0.5).astype(int),
                             (lstm_mp > 0.5).astype(int), average='macro', zero_division=0)
lstm_final_bin   = f1_score(lstm_b_true, (lstm_b_prob > 0.5).astype(int), average='binary', zero_division=0)
lstm_roc_auc     = roc_auc_score(lstm_b_true, lstm_b_prob) if len(np.unique(lstm_b_true)) > 1 else float('nan')

print(f'BiLSTM | macro-F1={lstm_final_macro:.4f} | bin-F1={lstm_final_bin:.4f} | ROC-AUC={lstm_roc_auc:.4f}')


## Classical-ML Baselines

These models are trained on TF-IDF + simple numeric features for comparison
against the transformer. They share a single `X_train`/`X_test` split so the
final comparison table is apples-to-apples (including the transformer, which
is re-scored on the same `X_test` rows in the comparison cell below).

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score
# -- Baseline setup: shared train/test split for ALL classical models -----------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

df_clean = df[df['toxic'].isin([0, 1])].copy()
df_clean['toxic'] = df_clean['toxic'].astype(int)

X = df_clean[['comment_text', 'word_count', 'text_length']]
y = df_clean['any_toxic']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

preprocessor = ColumnTransformer(transformers=[
    ('text', TfidfVectorizer(max_features=5000, stop_words='english'), 'comment_text'),
    ('num', MinMaxScaler(),   ['word_count', 'text_length']),
])

print('Preprocessing and transforming features...')
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)
print(f'Train: {X_train_processed.shape}  |  Test: {X_test_processed.shape}')


**Logistic Regression**

In [ ]:
param_grid = {'C': [0.1, 1, 10], 'solver': ['lbfgs', 'saga']}

grid_search = GridSearchCV(
    estimator=LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    param_grid=param_grid,
    scoring='f1_macro',
    cv=3,
    n_jobs=-1,
    verbose=1,
)

print('Starting Hyperparameter Tuning...')
grid_search.fit(X_train_processed, y_train)

print(f'\nBest Parameters Found: {grid_search.best_params_}')
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test_processed)
y_prob = best_model.predict_proba(X_test_processed)[:, 1]

print('\n--- Logistic Regression Evaluation ---')
print(classification_report(y_test, y_pred))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}')

plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.title('Logistic Regression - Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


**Linear SVM**

In [ ]:

svm_model = LinearSVC(class_weight='balanced', random_state=42, max_iter=2000)
svm_model.fit(X_train_processed, y_train)

y_svm_pred = svm_model.predict(X_test_processed)
print('--- Linear SVM Performance ---')
print(classification_report(y_test, y_svm_pred))


**Multinomial Naive Bayes**

In [ ]:

# MultinomialNB (not GaussianNB) is the correct choice for TF-IDF/count features,
# and works directly on the sparse matrix - no need to densify with .toarray(),
# which would blow up memory on anything bigger than a few thousand rows.
nb_model = MultinomialNB()
nb_model.fit(X_train_processed, y_train)

y_pred_nb = nb_model.predict(X_test_processed)

print('Accuracy:', accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))


**XGBoost**

In [ ]:

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
)
xgb_model.fit(X_train_processed, y_train)

y_pred_xgb = xgb_model.predict(X_test_processed)

print('Accuracy:', accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))


**Random Forest**

In [ ]:

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_processed, y_train)

y_pred_rf = rf_model.predict(X_test_processed)

print('Accuracy:', accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))


## Unified Model Comparison

Every model below is evaluated on the **same held-out val/test split**:
- Transformer models → val set produced by the stratified `train_test_split`
- BiLSTM → same val indices
- Classical ML models → `predict_batch()` run on the same val rows (or their own X_test, indicated per-row)

A genuine apples-to-apples comparison now replaces the previous version
where BERT's val predictions were incorrectly compared against a different
classical-ML test split of unrelated rows.


In [ ]:
# -- Unified leaderboard --------------------------------------------------------
from sklearn.metrics import accuracy_score, f1_score

all_results = []

# 1. Transformer results (from multi-transformer loop; empty if loop was skipped)
for name, res in transformer_results.items():
    all_results.append({
        'Model':      name,
        'Type':       'Transformer',
        'Macro-F1':   res['best_macro_f1'],
        'Binary-F1':  res['best_bin_f1'],
        'ROC-AUC':    res['roc_auc'],
    })

# 2. BiLSTM (skip gracefully if not trained)
if 'lstm_final_macro' in dir() or 'lstm_final_macro' in locals():
    all_results.append({
        'Model':     'BiLSTM',
        'Type':      'Deep Learning',
        'Macro-F1':  round(lstm_final_macro, 4),
        'Binary-F1': round(lstm_final_bin,   4),
        'ROC-AUC':   round(lstm_roc_auc,     4),
    })

# 3. Classical ML: score on the same val rows as the transformers.
#    Re-encode val rows using the already-fitted preprocessor.
val_X_classic = df.iloc[val_idx][['comment_text', 'word_count', 'text_length']]
val_X_proc    = preprocessor.transform(val_X_classic)
y_val_true    = (binary_labels[val_idx] > 0.5).astype(int)

classical_clfs = [
    ('Logistic Regression', best_model),
    ('Linear SVM',          svm_model),
    ('Naive Bayes',         nb_model),
    ('Random Forest',       rf_model),
    ('XGBoost',             xgb_model),
]

for clf_name, clf in classical_clfs:
    y_pred_c = clf.predict(val_X_proc)
    try:
        y_prob_c = clf.predict_proba(val_X_proc)[:, 1]
        roc_c = roc_auc_score(y_val_true, y_prob_c)
    except Exception:
        roc_c = float('nan')

    all_results.append({
        'Model':     clf_name,
        'Type':      'Classical ML',
        'Macro-F1':  round(f1_score(y_val_true, y_pred_c, average='macro',  zero_division=0), 4),
        'Binary-F1': round(f1_score(y_val_true, y_pred_c, average='binary', zero_division=0), 4),
        'ROC-AUC':   round(roc_c, 4),
    })

# 4. DeBERTa checkpoint (if saved)
try:
    bert_val_df  = predict_batch(df.iloc[val_idx]['comment_text'].tolist())
    y_bert_pred  = bert_val_df['toxic'].astype(int).values
    y_bert_prob  = bert_val_df['score'].values
    all_results.append({
        'Model':     'DeBERTa (checkpoint)',
        'Type':      'Transformer',
        'Macro-F1':  round(f1_score(y_val_true, y_bert_pred, average='macro',  zero_division=0), 4),
        'Binary-F1': round(f1_score(y_val_true, y_bert_pred, average='binary', zero_division=0), 4),
        'ROC-AUC':   round(roc_auc_score(y_val_true, y_bert_prob) if len(np.unique(y_val_true)) > 1 else float('nan'), 4),
    })
except Exception as _e:
    print(f'[Info] DeBERTa checkpoint scoring skipped: {_e}')

compare_df = (
    pd.DataFrame(all_results)
    .sort_values('Binary-F1', ascending=False)
    .reset_index(drop=True)
)
compare_df.index += 1

print('\n========== Model Leaderboard (same held-out split) ==========\n')
print(compare_df.to_string())


In [ ]:
compare_df.style.format({
    'Macro-F1':  '{:.4f}',
    'Binary-F1': '{:.4f}',
    'ROC-AUC':   '{:.4f}',
}).background_gradient(subset=['Binary-F1', 'ROC-AUC'], cmap='Greens')


In [ ]:
# -- Visual leaderboard bar chart -----------------------------------------------
fig, ax = plt.subplots(figsize=(12, 6))

type_colors = {'Transformer': '#4C9BE8', 'Deep Learning': '#E8A54C', 'Classical ML': '#9BE84C'}
bar_colors  = [type_colors.get(t, '#aaaaaa') for t in compare_df['Type']]

bars = ax.barh(compare_df['Model'], compare_df['Binary-F1'], color=bar_colors, edgecolor='white')
for bar, val in zip(bars, compare_df['Binary-F1']):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

ax.set_xlabel('Binary F1-Score')
ax.set_title('Model Leaderboard — All Architectures (same held-out split)', fontweight='bold', fontsize=13)
ax.invert_yaxis()
ax.set_xlim(0, compare_df['Binary-F1'].max() + 0.12)

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=c, label=t) for t, c in type_colors.items()]
ax.legend(handles=legend_handles, loc='lower right', fontsize=9)

plt.tight_layout()
plt.show()


## Inference Demonstration

In [ ]:
# -- Inference demonstration (runs only when a checkpoint exists) ---------------
import os as _os

_ckpt_ok = _os.path.exists(_os.path.join(CFG.save_dir, 'config.json'))

if _ckpt_ok:
    model = load_model_from_checkpoint(CFG.save_dir)
    sample_text = "You are an idiot and I hate you."
    prediction_result = predict_toxicity(sample_text)

    print('\n--- Inference Result ---')
    print(f'Input Text: {sample_text}')
    print(f"Toxic : {prediction_result['toxic']}")
    print(f"Score : {prediction_result['score']:.4f}")
    print('Labels:')
    for label, info in prediction_result['labels'].items():
        print(f"  {'⚠' if info['flagged'] else ' '} {label:15} {info['prob']:.4f}")
else:
    print('[Info] No saved checkpoint found in', CFG.save_dir)
    print('       Run the training loop (cell 13) first, then re-run this cell.')


## Export

One canonical save + zip (+ optional Google Drive copy). This replaces the
earlier tail of ~15 cells that repeatedly saved the model to different paths
(`Data Science Project`, `toxicity_model`, `/content/toxicity_model`, ...),
re-checked `num_labels` four separate times, and printed `model.classifier` /
`os.getcwd()` for no functional reason.

In [ ]:
# Re-confirm the artifacts in the canonical save_dir
print('Save directory:', CFG.save_dir)
print('Files:', os.listdir(CFG.save_dir))

ckpt_path = os.path.join(CFG.save_dir, 'best_checkpoint.pt')
if os.path.exists(ckpt_path):
    size_mb = os.path.getsize(ckpt_path) / (1024 * 1024)
    print(f'best_checkpoint.pt size: {size_mb:.2f} MB')


In [ ]:
# Zip the save directory for download
import shutil
shutil.make_archive('Toxic_Model', 'zip', CFG.save_dir)
print('Created Toxic_Model.zip')


In [ ]:
# Colab-only: download the zip / mount Drive. Safe to skip outside Colab.
try:
    from google.colab import files, drive

    files.download('Toxic_Model.zip')

    # Uncomment to also persist a copy to Google Drive:
    # drive.mount('/content/drive')
    # shutil.copytree(CFG.save_dir, '/content/drive/MyDrive/toxicity_model', dirs_exist_ok=True)
    # print('Model copied to Google Drive.')
except ImportError:
    print('Not running in Google Colab - skipping download/drive steps. '
          f'Your model and zip are available locally at "{CFG.save_dir}" and "Toxic_Model.zip".')
